In [ ]:
# ================================================================
#   Cell × Trait Association Analysis Pipeline
#   - Load SCANPY AnnData
#   - Load LDSC-derived Z-score matrix
#   - Harmonize trait labels
#   - Rank and annotate significance
#   - Plot heatmap of cluster–trait associations
#   - Map trait Z-scores back to scRNA AnnData
#   - Plot spatial Z-score map per sample
# ================================================================

import os
import re
import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm, Normalize
from matplotlib.lines import Line2D
from pathlib import Path

# ================================================================
# 1. Load scRNA-seq data
# ================================================================
scrna = sc.read_h5ad("./data/scrna_gw34_minimal_subCluster_spatial.h5ad")
print(scrna)


# ================================================================
# 2. Load combined Z-score table (cell × trait)
# ================================================================
z_path = "./ldsc/scatac/_combined/combined_z.tsv"
df = pd.read_csv(z_path, sep="\t")

# Remove unnamed columns
df = df.loc[:, ~df.columns.str.contains(r"^Unnamed")]

# Ensure first col = 'cell'
if "cell" not in df.columns:
    df = df.rename(columns={df.columns[0]: "cell"})

# Wide format: rows = subClusters, cols = traits
zmat = df.set_index("cell")

# Remove ".sumstats" suffix if present
zmat = zmat.rename(columns=lambda c: c.replace(".sumstats", ""))


# ================================================================
# 3. Map original trait IDs → human-readable names
# ================================================================
trait_map = {
    "OCD": "OCD",
    "SCZ": "SCZ",
    "MDD": "MDD",
    "ASD": "ASD",
    "BIP": "Bipolar_disorder",
    "insomnia_ldsc": "Insomnia",
    "ADHD": "ADHD",
}

zmat = zmat.rename(columns=lambda c: trait_map.get(c, c))

# In case mapping or suffix removal causes duplicated columns
zmat = zmat.T.groupby(level=0).mean().T


# ================================================================
# 4. Sort rows/cols by max absolute Z-score
# ================================================================
zmat = zmat[zmat.abs().max(axis=0).sort_values(ascending=False).index]
zmat = zmat.loc[zmat.abs().max(axis=1).sort_values(ascending=False).index]

cells = zmat.index.tolist()
traits = zmat.columns.tolist()
Z = zmat.values.astype(float)


# ================================================================
# 5. Identify significant signals & top-3 markers
# ================================================================
thr = 3.0
signif = np.abs(Z) >= thr

rank_marks = np.empty(Z.shape, dtype=object)
rank_marks[:] = ""

for j, col in enumerate(traits):
    col_vals = zmat[col].abs().replace([np.inf, -np.inf], np.nan)
    top_idx = col_vals.nlargest(3).dropna().index.tolist()

    marks = { "***": top_idx[0] if len(top_idx) > 0 else None,
              "**": top_idx[1] if len(top_idx) > 1 else None,
              "*":  top_idx[2] if len(top_idx) > 2 else None }

    for m, name in marks.items():
        if name is None:
            continue
        i = cells.index(name)
        rank_marks[i, j] = m


# ================================================================
# 6. Heatmap visualization
# ================================================================
fig_w = max(8, len(traits) * 0.5)
fig_h = max(8, len(cells) * 0.30)
fig, ax = plt.subplots(figsize=(fig_w, fig_h))

norm = TwoSlopeNorm(vmin=np.nanmin(Z), vcenter=0.0, vmax=np.nanmax(Z))
im = ax.imshow(Z, aspect="auto", cmap="bwr", norm=norm)

ax.set_xticks(range(len(traits)))
ax.set_xticklabels(traits, rotation=60, ha="right", fontsize=8)
ax.set_yticks(range(len(cells)))
ax.set_yticklabels(cells, fontsize=8)

cbar = plt.colorbar(im, ax=ax, fraction=0.02, pad=0.02)
cbar.set_label("Z-score", rotation=90)

# Significant ▲▽ markers
ys, xs = np.where(signif)
for i, j in zip(ys, xs):
    val = Z[i, j]
    if np.isnan(val):
        continue
    if val >= thr:
        ax.scatter(j, i, marker="^", s=60, color="black")
    elif val <= -thr:
        ax.scatter(j, i, marker="v", s=60, color="black")

# Top-3 stars
for i in range(Z.shape[0]):
    for j in range(Z.shape[1]):
        mark = rank_marks[i, j]
        if not mark:
            continue
        color = "black" if signif[i, j] else "dimgray"
        ax.text(j, i, mark, ha="center", va="center",
                fontsize=8, color=color)

# Grid lines
ax.set_xticks(np.arange(-0.5, len(traits), 1), minor=True)
ax.set_yticks(np.arange(-0.5, len(cells), 1), minor=True)
ax.grid(which="minor", color="white", linestyle='-', linewidth=0.5)

# Legend
legend_elems = [
    Line2D([0], [0], marker='^', color='w',
           label='Positive significant (|Z| ≥ 3)',
           markerfacecolor='black', markersize=8),
    Line2D([0], [0], marker='v', color='w',
           label='Negative significant (|Z| ≥ 3)',
           markerfacecolor='black', markersize=8),
    Line2D([0], [0], marker='s', color='w',
           label='Top-3 per trait',
           markerfacecolor='lightgray', markeredgecolor='lightgray', markersize=8),
]
ax.legend(handles=legend_elems, loc='upper left', bbox_to_anchor=(1.02, 1.02))

ax.set_title("Cell × Trait Association (Z-score Heatmap)")
plt.tight_layout()
plt.show()


# ================================================================
# 7. Attach trait Z-scores back to scRNA AnnData
# ================================================================
def _sanitize_colname(name: str) -> str:
    safe = re.sub(r'[^0-9A-Za-z_]+', '_', str(name).strip())
    safe = re.sub(r'_+', '_', safe).strip('_')
    return f"Z__{safe}"

def _canon_key(x: str, lower=True, strip=True, hy2us=True):
    s = str(x)
    if strip:
        s = s.strip()
    if hy2us:
        s = s.replace('-', '_')
    if lower:
        s = s.lower()
    s = re.sub(r'\s+', ' ', s)
    return s

def attach_z_to_scrna(scrna, zmat, key_obs="subCluster",
                      fillna=None, store_to_obsm=True,
                      obsm_key="Z_diseases", lower_match=True):

    assert key_obs in scrna.obs.columns, f"{key_obs} not found in scrna.obs"

    z = zmat.copy()
    z.index = z.index.map(lambda s: _canon_key(s, lower=lower_match))

    obs_key = scrna.obs[key_obs].astype(str).map(lambda s: _canon_key(s, lower=lower_match))

    if fillna == "colmean":
        z = z.apply(lambda s: s.fillna(s.mean()), axis=0)
    elif fillna is not None:
        z = z.fillna(fillna)

    mapped_cols = []
    for d in z.columns:
        cname = _sanitize_colname(d)
        scrna.obs[cname] = obs_key.map(z[d]).astype(float)
        mapped_cols.append(cname)

    if store_to_obsm:
        scrna.obsm[obsm_key] = scrna.obs[mapped_cols].to_numpy()
        scrna.uns[f"{obsm_key}__cols"] = mapped_cols

    print(f"[OK] Added {len(mapped_cols)} Z-score columns to scrna.obs")
    return mapped_cols

mapped_cols = attach_z_to_scrna(
    scrna, zmat,
    key_obs="subCluster",
    fillna=None,
    obsm_key="Z_diseases"
)
print("Z__SCZ in obs? ", "Z__SCZ" in scrna.obs.columns)


# ================================================================
# 8. Spatial Z-score Visualization
# ================================================================
class SymmetricPowerNorm(Normalize):
    """Symmetric power normalization for mid-range saturation."""
    def __init__(self, vmax=5.0, gamma=0.7, clip=False):
        super().__init__(vmin=-vmax, vmax=vmax, clip=clip)
        self.vmax = vmax
        self.gamma = gamma

    def __call__(self, value, clip=None):
        v = np.asarray(value, dtype=float)
        out = np.full_like(v, np.nan)
        mask = np.isfinite(v)
        if not mask.any():
            return out
        vv = v[mask]
        s = np.sign(vv)
        a = np.clip(np.abs(vv) / self.vmax, 0, 1) ** self.gamma
        out[mask] = 0.5 + 0.5 * s * a
        return out

# Get spatial coordinates
if "spatial" in scrna.obsm:
    X = np.asarray(scrna.obsm["spatial"])[:, :2]
else:
    raise ValueError("No spatial coordinates found.")

finite_mask = np.isfinite(X).all(axis=1)

sample_to_plot = scrna.obs["sample_id"].value_counts().idxmax()
sel = (scrna.obs["sample_id"].astype(str) == sample_to_plot) & finite_mask

disease_col = "Z__SCZ"
x, y = X[sel, 0], X[sel, 1]
z = scrna.obs.loc[sel, disease_col].astype(float).to_numpy()

plt.figure(figsize=(7, 7), facecolor="white")
norm = SymmetricPowerNorm(vmax=5.0, gamma=0.7)
scplt = plt.scatter(x, y, c=z, s=3, cmap="coolwarm", norm=norm, alpha=0.9)
plt.gca().invert_yaxis()
plt.title(f"{sample_to_plot} — SCZ Z-score")
plt.colorbar(scplt, label="SCZ Z-score")
plt.axis("equal")
plt.tight_layout()
plt.show()